In [ ]:
import pandas as pd
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import STL
from tqdm import tqdm
import numpy as np

In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.0 MB/s eta 0:00:00


In [ ]:
df = pd.read_csv("df_final.csv", parse_dates=["timestamp"])

houses = [col for col in df.columns if col.startswith("house_")]

In [ ]:
periods = {
    "недельная": 336,
    "месячная":  1488,
    "годовая":   17520,
}

In [ ]:
stl_results = {}

for period_name, period in periods.items():
    results = []

    for house in tqdm(houses, desc=f"STL {period_name}"):
        s = df.set_index("timestamp")[house]

        stl = STL(s, period=period, robust=True)
        res = stl.fit()

        total_var = res.trend.var() + res.seasonal.var() + res.resid.var()

        results.append({
            "Дом": house,
            "Дисперсия тренда": round(res.trend.var(), 2),
            "Дисперсия сезонности": round(res.seasonal.var(), 2),
            "Дисперсия остатка": round(res.resid.var(), 2),
            "Доля сезонности, %": round(res.seasonal.var() / total_var * 100, 1)
        })

    stl_results[period_name] = pd.DataFrame(results)
    print()
    print(f"{period_name.upper()} Сезонность:")
    print(stl_results[period_name].to_string(index=False))
    print()

STL недельная: 100%|██████████| 8/8 [07:05<00:00, 53.21s/it]



НЕДЕЛЬНАЯ Сезонность:
    Дом  Дисперсия тренда  Дисперсия сезонности  Дисперсия остатка  Доля сезонности, %
house_1             39.93                301.77              14.40                84.7
house_2              7.15                 46.87               3.46                81.5
house_3              3.64                 19.07               2.18                76.6
house_4             22.11                118.47               6.04                80.8
house_5             12.01                 45.15               3.35                74.6
house_6             80.72                513.53              24.22                83.0
house_7             66.72                742.91              25.65                88.9
house_8             13.90                 80.55               4.45                81.4



STL месячная: 100%|██████████| 8/8 [30:50<00:00, 231.37s/it]



МЕСЯЧНАЯ Сезонность:
    Дом  Дисперсия тренда  Дисперсия сезонности  Дисперсия остатка  Доля сезонности, %
house_1             31.17                284.79              39.78                80.1
house_2              5.93                 44.50               7.41                76.9
house_3              3.10                 17.84               3.69                72.4
house_4             18.08                112.83              16.41                76.6
house_5             10.31                 43.38               6.32                72.3
house_6             63.66                477.86              61.72                79.2
house_7             54.61                720.73              80.08                84.3
house_8             11.68                 78.46               8.69                79.4



STL годовая: 100%|██████████| 8/8 [6:02:59<00:00, 2722.38s/it]


ГОДОВАЯ Сезонность:
    Дом  Дисперсия тренда  Дисперсия сезонности  Дисперсия остатка  Доля сезонности, %
house_1              0.29                357.85               0.00                99.9
house_2              0.03                 57.36               0.01                99.9
house_3              0.01                 24.80               0.00                99.9
house_4              0.67                143.62               0.01                99.5
house_5              0.00                 60.49               0.00               100.0
house_6              4.83                615.34               0.08                99.2
house_7              0.36                826.17               0.00               100.0
house_8              0.10                 99.58               0.00                99.9



In [ ]:
#Чтобы заново не запускать расчет
# недельная сезонность
data_weekly = {
    "дом": ["house_1", "house_2", "house_3", "house_4", "house_5", "house_6", "house_7", "house_8"],
    "дисперсия_тренда": [39.93, 7.15, 3.64, 22.11, 12.01, 80.72, 66.72, 13.90],
    "дисперсия_сезонности": [301.77, 46.87, 19.07, 118.47, 45.15, 513.53, 742.91, 80.55],
    "дисперсия_остатка": [14.40, 3.46, 2.18, 6.04, 3.35, 24.22, 25.65, 4.45],
    "доля_сезонности": [84.7, 81.5, 76.6, 80.8, 74.6, 83.0, 88.9, 81.4],
    "период": "недельная"
}

# месячная сезонность
data_monthly = {
    "дом": ["house_1", "house_2", "house_3", "house_4", "house_5", "house_6", "house_7", "house_8"],
    "дисперсия_тренда": [31.17, 5.93, 3.10, 18.08, 10.31, 63.66, 54.61, 11.68],
    "дисперсия_сезонности": [284.79, 44.50, 17.84, 112.83, 43.38, 477.86, 720.73, 78.46],
    "дисперсия_остатка": [39.78, 7.41, 3.69, 16.41, 6.32, 61.72, 80.08, 8.69],
    "доля_сезонности": [80.1, 76.9, 72.4, 76.6, 72.3, 79.2, 84.3, 79.4],
    "период": "месячная"
}

# годовая сезонность
data_yearly = {
    "дом": ["house_1", "house_2", "house_3", "house_4", "house_5", "house_6", "house_7", "house_8"],
    "дисперсия_тренда": [0.29, 0.03, 0.01, 0.67, 0.00, 4.83, 0.36, 0.10],
    "дисперсия_сезонности": [357.85, 57.36, 24.80, 143.62, 60.49, 615.34, 826.17, 99.58],
    "дисперсия_остатка": [0.00, 0.01, 0.00, 0.01, 0.00, 0.08, 0.00, 0.00],
    "доля_сезонности": [99.9, 99.9, 99.9, 99.5, 100.0, 99.2, 100.0, 99.9],
    "период": "годовая"
}

In [ ]:
df_stl = pd.concat([
    pd.DataFrame(data_weekly),
    pd.DataFrame(data_monthly),
    pd.DataFrame(data_yearly)
], ignore_index=True)

df_stl.to_csv("results_stl.csv", index=False)

In [ ]:
houses = ["house_1", "house_2", "house_3", "house_4",
          "house_5", "house_6", "house_7", "house_8"]

colors = {
    "house_1": "#1f77b4", "house_2": "#ff7f0e",
    "house_3": "#2ca02c", "house_4": "#d62728",
    "house_5": "#9467bd", "house_6": "#8c564b",
    "house_7": "#e377c2", "house_8": "#7f7f7f"
}

period_colors = {
    "недельная": "#1f77b4",
    "месячная":  "#ff7f0e",
    "годовая":   "#d62728"
}

In [ ]:
fig1 = go.Figure()

for period in ["недельная", "месячная", "годовая"]:
    data = df_stl[df_stl["период"] == period]
    fig1.add_trace(go.Bar(
        x=data["дом"],
        y=data["доля_сезонности"],
        name=period,
        marker_color=period_colors[period]
    ))

fig1.update_layout(
    title="Доля сезонности по периодам (STL)",
    xaxis_title="Дом",
    yaxis_title="Доля сезонности, %",
    barmode="group",
    width=700, height=400,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig1.show()
fig1.write_image("stl_01_share.png", scale=2)

In [ ]:
fig2 = go.Figure()

for period in ["недельная", "месячная", "годовая"]:
    data = df_stl[df_stl["период"] == period]
    fig2.add_trace(go.Bar(
        x=data["дом"],
        y=data["дисперсия_сезонности"],
        name=period,
        marker_color=period_colors[period]
    ))

fig2.update_layout(
    title="Дисперсия сезонности по периодам (STL)",
    xaxis_title="Дом",
    yaxis_title="Дисперсия сезонности",
    barmode="group",
    width=700, height=400,
    font=dict(size=11),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig2.show()
fig2.write_image("stl_02_variance.png", scale=2)

In [ ]:
for period in ["недельная", "месячная", "годовая"]:
    data = df_stl[df_stl["период"] == period]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=data["дом"], y=data["дисперсия_тренда"],
        name="тренд", marker_color="#1f77b4"
    ))
    fig.add_trace(go.Bar(
        x=data["дом"], y=data["дисперсия_сезонности"],
        name="сезонность", marker_color="#ff7f0e"
    ))
    fig.add_trace(go.Bar(
        x=data["дом"], y=data["дисперсия_остатка"],
        name="остаток", marker_color="#2ca02c"
    ))

    fig.update_layout(
        title=f"Декомпозиция дисперсии - {period} сезонность (STL)",
        xaxis_title="Дом",
        yaxis_title="Дисперсия",
        barmode="group",
        width=700, height=400,
        font=dict(size=11),
        margin=dict(l=50, r=20, t=40, b=40),
        legend=dict(font=dict(size=9))
    )
    fig.show()
    fig.write_image(f"stl_03_{period}.png", scale=2)

Недельная сезонность:
Недельная составляющая объясняет от 74.6% (дом 5) до 88.9% (дом 7) общей вариации временного ряда. Это означает что зная день недели и время суток можно объяснить подавляющую часть колебаний электропотребления. Наблюдается чёткая закономерность - чем больше квартир в доме, тем выше доля недельной сезонности. Дом 7 (471 квартира) показывает максимальную долю 88.9%, тогда как дома 5 и 3 с наименьшим количеством квартир (127 и 124) имеют минимальные значения 74.6% и 76.6% соответственно. Это подтверждает эффект агрегации - в крупных домах случайные отклонения отдельных квартир усредняются, делая суммарное потребление более регулярным и предсказуемым.
Месячная сезонность:
При увеличении периода до месяца доля сезонности незначительно снижается - от 72.3% (дом 5) до 84.3% (дом 7). Снижение объясняется тем что месячная составляющая включает в себя переходные периоды между сезонами года, которые вносят дополнительную нестабильность. Дисперсия остатка при месячной декомпозиции заметно выше чем при недельной - это говорит о том что месячный период менее точно описывает структуру ряда по сравнению с недельным. Тем не менее высокая доля сезонности сохраняется для всех домов.
Годовая сезонность:
Годовая составляющая демонстрирует исключительно высокую долю сезонности - от 99.2% (дом 6) до 100% (дома 5 и 7). Дисперсия остатка практически равна нулю для всех домов - модель идеально описывает данные на годовом горизонте. Дисперсия тренда также минимальна, что свидетельствует об отсутствии значимых долгосрочных изменений уровня потребления за рассматриваемый период. Этот результат подтверждает что электропотребление жилых домов подчиняется устойчивому годовому циклу - зимой потребление выше, летом ниже, и этот паттерн воспроизводится из года в год с высокой точностью.
Заключение:
STL-декомпозиция выявила три уровня сезонности в данных электропотребления жилых домов. Недельная и месячная сезонность объясняют 72-89% вариации и отражают поведенческие паттерны жильцов - суточные и недельные циклы активности. Годовая сезонность объясняет 99-100% вариации и отражает климатические факторы - температуру наружного воздуха и световой день. Полученные результаты подтверждают целесообразность включения сезонных признаков всех трёх уровней в модели прогнозирования, а также обосновывают использование данных за полный год в обучающей выборке для корректного учёта годового цикла. Эффект агрегации - более высокая предсказуемость крупных домов - имеет практическое значение для электросетевых компаний при планировании нагрузки на уровне жилых кварталов.